# TP 2 — Cibler une campagne téléphonique bancaire

**Classification — niveau débutant — environ 2 h — énoncé**

## Mise en situation

Tu es data scientist dans une banque de détail. Le service marketing lance chaque trimestre
une campagne téléphonique pour vendre un **dépôt à terme** (un placement bloqué, rémunéré).
Le bilan de la dernière campagne est le suivant : 45 000 clients appelés, **un peu moins de
12 % ont souscrit**. Les 88 % restants ont coûté du temps de conseiller, et agacé des
clients.

Le directeur marketing te pose sa question :

> **À partir de ce que la banque sait déjà de ses clients (âge, métier, solde du compte,
> crédits en cours, historique des campagnes précédentes), peut-on prédire qui va souscrire,
> et donc n'appeler que les clients les plus prometteurs ?**

Ce qu'on cherche à prédire est une **catégorie** (souscrit / ne souscrit pas) : c'est un
problème de **classification binaire**.

Les données sont publiques : ce sont celles d'une banque portugaise, mises à disposition par
l'UCI Machine Learning Repository et téléchargées ici depuis OpenML. Une ligne = un client
appelé lors de la campagne.

> Le chargement télécharge les données la première fois (connexion internet nécessaire),
> puis les met en cache sur ta machine.

## Ce que tu vas faire

| Étape | Ce que tu apprends |
|---|---|
| 1. Manipuler les données | charger, inspecter un jeu mêlant nombres et catégories |
| 2. Explorer les données | lire un taux de souscription, repérer les segments qui répondent |
| 3. Préparer les données | construire la cible binaire et éliminer une variable qui triche |
| 4. Nettoyer les données | les `unknown`, et les valeurs codées en douce comme `-1` |
| 5. Ton premier modèle | un `Pipeline` qui encode les catégories et entraîne une régression logistique |
| 6. Évaluer le modèle | pourquoi l'accuracy ment ici : matrice de confusion, precision, recall, AUC |
| 7. Cross-validation | obtenir une estimation fiable et stable |
| 8. Verdict final | évaluer une fois sur le jeu de test et chiffrer le gain pour le marketing |

## Comment travailler

- Les cellules **À TOI DE JOUER** contiennent une consigne en commentaires : c'est à toi
  d'écrire le code en dessous.
- Respecte les noms de variables donnés dans les consignes : les exercices suivants les
  réutilisent. Si une variable manque, la cellule d'après ne tournera pas.
- Exécute les cellules **dans l'ordre**, de haut en bas.
- Chaque exercice se termine par un encadré **Ce que tu devrais observer** : c'est ton
  auto-correction. Si tu ne retrouves pas ces chiffres, relis ton code avant de continuer.
- Le corrigé complet est dans `02_tp_classification_campagne_bancaire_corrige.ipynb`.

In [ ]:
# Cellule a executer une fois, au debut du TP
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
print("Librairies chargées.")

---
## Étape 1 — Manipuler les données

### Le dictionnaire des données

**Le client**

| Colonne | Signification |
|---|---|
| `age` | âge du client |
| `job` | catégorie professionnelle (12 modalités) |
| `marital` | situation familiale : `married`, `single`, `divorced` |
| `education` | niveau d'études : `primary`, `secondary`, `tertiary`, `unknown` |
| `default` | le client a-t-il un incident de paiement en cours ? |
| `balance` | solde annuel moyen du compte, en euros |
| `housing` | a-t-il un crédit immobilier en cours ? |
| `loan` | a-t-il un crédit à la consommation en cours ? |

**Le dernier appel de la campagne en cours**

| Colonne | Signification |
|---|---|
| `contact` | moyen de contact : `cellular`, `telephone`, `unknown` |
| `day` / `month` | jour et mois du dernier appel |
| `duration` | **durée du dernier appel, en secondes** |

**L'historique**

| Colonne | Signification |
|---|---|
| `campaign` | nombre d'appels passés à ce client pendant cette campagne |
| `pdays` | nombre de jours depuis le dernier contact d'une campagne précédente (**-1 = jamais contacté**) |
| `previous` | nombre de contacts avant cette campagne |
| `poutcome` | résultat de la campagne précédente : `success`, `failure`, `other`, `unknown` |
| `y` | **la cible** : le client a-t-il souscrit ? `yes` / `no` |

Garde un œil sur `duration` : cette colonne est un piège, et tu le découvriras à l'étape 2.

### Exercice 1.1 — Charger et regarder

Charge le jeu de données, puis affiche ses dimensions, ses trois premières lignes et le type
de chaque colonne.

Indice : `fetch_openml(data_id=44234, as_frame=True).frame` renvoie un DataFrame pandas
complet.

In [ ]:
# A TOI DE JOUER
# 1. Importe fetch_openml depuis sklearn.datasets
# 2. Charge les donnees dans une variable nommee clients (un DataFrame)
# 3. Affiche ses dimensions, ses 3 premieres lignes, et les types de ses colonnes

**Ce que tu devrais observer** : 45 211 clients et 17 colonnes. Contrairement au TP de
régression, **toutes les colonnes ne sont pas numériques** : `job`, `marital`, `month`,
`poutcome`... contiennent du texte. Ce sont des **variables catégorielles**, et un modèle
mathématique ne sait pas les additionner. Il faudra les transformer en nombres à l'étape 5.

### Exercice 1.2 — Interroger le tableau

1. Combien de clients ont souscrit, et quelle proportion cela représente-t-il ?
   (`clients["y"].value_counts()` puis `value_counts(normalize=True)`)
2. Quelles sont les statistiques des colonnes numériques ? (`.describe()`)
3. Quelles sont les 12 catégories professionnelles, et leurs effectifs ?

In [ ]:
# A TOI DE JOUER
# 1. Repartition de la cible y, en effectif puis en proportion
# 2. Statistiques descriptives des colonnes numeriques
# 3. Effectif de chaque categorie professionnelle

**Ce que tu devrais observer** : 5 289 clients ont souscrit sur 45 211, soit **11,7 %**. Les
deux classes sont donc très **déséquilibrées** : pour un « oui », il y a presque huit
« non ». Retiens ce chiffre de 11,7 %, il sera la clé de l'étape 6.

Côté variables numériques, deux colonnes surprennent déjà : `balance` descend à -8 019 euros
(des comptes à découvert) et `pdays` vaut -1 au minimum, ce qui n'a aucun sens pour un
nombre de jours. On s'en occupera à l'étape 4.

---
## Étape 2 — Explorer les données

**Question de réflexion :** avec 11,7 % de souscriptions en moyenne, qu'est-ce qui serait
utile au marketing ? Connaître les segments dont le taux **s'écarte** franchement de cette
moyenne.

C'est la logique de l'exploration en classification : on compare toujours le taux d'un
groupe au **taux global**.

### Exercice 2.1 — Le taux de souscription, segment par segment

Pour `job`, `housing` et `poutcome`, calcule le pourcentage de clients qui ont souscrit dans
chaque modalité, trié par ordre décroissant.

Indice : `clients.groupby("job")["y"].apply(lambda s: (s == "yes").mean() * 100)`, puis
`.sort_values(ascending=False).round(1)`.

In [ ]:
# A TOI DE JOUER
# Pour chacune des colonnes job, housing et poutcome :
# affiche le taux de souscription (en %) de chaque modalite, trie par ordre decroissant

**Ce que tu devrais observer**, et ce que ça raconte côté métier :

- **`job`** : les étudiants (28,7 %) et les retraités (22,8 %) souscrivent deux à trois fois
  plus que la moyenne, les ouvriers (`blue-collar`, 7,3 %) deux fois moins. Un produit
  d'épargne bloquée parle à ceux qui ont du temps devant eux ou un capital à placer.
- **`housing`** : 16,7 % sans crédit immobilier contre 7,7 % avec. Logique : quand on
  rembourse un prêt, on n'immobilise pas son épargne.
- **`poutcome`** : **64,7 %** des clients qui avaient déjà dit oui lors d'une campagne
  précédente disent oui à nouveau. C'est de très loin le signal le plus fort du jeu de
  données. En marketing, c'est une règle connue : le meilleur prospect est un client.

Prudence quand même : un taux calculé sur peu de clients est peu fiable. Regarde toujours
l'effectif du segment (`value_counts`) avant de tirer une conclusion.

### Exercice 2.2 — L'âge, et la forme des relations

Trace deux graphiques :

1. La distribution de l'âge des clients qui ont souscrit, superposée à celle des autres
   (`sns.histplot(data=clients, x="age", hue="y", ...)`).
2. Le taux de souscription par tranche d'âge. Crée d'abord la tranche avec
   `pd.cut(clients["age"], bins=[17, 30, 40, 50, 60, 100])`, puis applique le même
   `groupby` qu'à l'exercice précédent.

In [ ]:
# A TOI DE JOUER
# 1. Histogramme de l'age, separe selon la cible y
# 2. Taux de souscription par tranche d'age (pd.cut avec les bornes 17, 30, 40, 50, 60, 100)

**Ce que tu devrais observer** : le taux de souscription dessine un **U** — 16,3 % chez les
moins de 30 ans, puis un creux à 10,2 %, 9,1 % et 10,1 % entre 30 et 60 ans, et enfin
**42,3 % au-delà de 60 ans**. Les seniors sont de loin la cible la plus rentable de cette
campagne, ce qui rejoint les 22,8 % des retraités vus à l'exercice précédent.

Vérifie quand même l'effectif de cette dernière tranche : 1 188 clients seulement, contre
17 687 pour la tranche 30-40 ans. Le taux reste solide sur plus de mille personnes, mais le
réflexe doit être automatique — un taux spectaculaire sur trente clients ne vaut rien.

Retiens aussi la leçon de lecture : une relation en U **n'apparaît pas** dans un coefficient
de corrélation, qui ne mesure qu'une tendance « plus de ceci, plus de cela ». Un graphique
voit ce qu'un chiffre unique manque.

### Exercice 2.3 — Le piège : la colonne `duration`

Compare la durée moyenne des appels selon que le client a souscrit ou non, puis pose-toi la
question qui suit — elle compte plus que le code.

In [ ]:
# A TOI DE JOUER
# Duree moyenne (et mediane) du dernier appel, selon que le client a souscrit ou non

**Ce que tu devrais observer** : un appel qui se conclut par une souscription dure en moyenne
**537 secondes** (9 minutes), contre **221 secondes** (moins de 4 minutes) sinon. Une
variable qui sépare aussi bien les deux classes ressemble à une aubaine.

**Question de réflexion, la plus importante du TP :** à quel moment connaît-on la valeur de
`duration` ?

Réponse : **après l'appel**. Or le modèle doit désigner qui appeler, donc **avant**. Une
variable qui n'existe pas au moment où l'on doit décider s'appelle une **fuite de données**
(*data leakage*). Elle donne des scores spectaculaires en TP et un modèle inutilisable en
production — c'est l'une des erreurs les plus fréquentes chez les débutants, et elle ne se
détecte pas dans les chiffres : elle se détecte en se demandant **quand** chaque information
devient disponible.

À l'étape 3, on jettera donc `duration`.

---
## Étape 3 — Préparer les données

Deux gestes ici :

1. **Construire la cible binaire.** Les modèles de scikit-learn attendent des nombres :
   `yes` devient 1, `no` devient 0. Par convention, **1 est la classe qui nous intéresse**
   (ici la souscription), celle dont on parlera comme la classe « positive ».
2. **Choisir les variables explicatives.** Tout ce qui sera connu avant l'appel — donc tout,
   sauf `y` et sauf `duration`.

### Exercice 3.1 — Cible binaire et variables explicatives

1. Crée une copie du DataFrame, nommée `donnees`, avec une colonne `souscrit` valant 1 ou 0.
2. Construis `y` (la colonne `souscrit`) et `X` (toutes les colonnes sauf `y`, `souscrit` et
   `duration`).
3. Vérifie les dimensions et le taux de 1 dans `y`.

Indice : `(donnees["y"] == "yes").astype(int)` transforme un booléen en 0/1.

In [ ]:
# A TOI DE JOUER
# 1. donnees = une copie de clients, avec une colonne souscrit valant 1 (yes) ou 0 (no)
# 2. y = donnees["souscrit"]  et  X = donnees sans les colonnes y, souscrit et duration
# 3. Affiche les dimensions de X et y, et la proportion de 1 dans y

**Ce que tu devrais observer** : `X` contient 45 211 lignes et **15 colonnes** (les 16
variables moins `duration`), et `y` vaut 1 pour 11,7 % des clients.

Tu viens de renoncer volontairement à la variable la plus prédictive du jeu de données. Ce
n'est pas un sacrifice, c'est la condition pour que le modèle serve à quelque chose.

---
## Étape 4 — Nettoyer les données

**Question de réflexion :** `clients.isna().sum()` ne renvoie que des zéros. Peut-on en
conclure qu'il n'y a pas de valeur manquante ?

Non. Une information absente peut être **déguisée** : un `"unknown"` dans une colonne texte,
un `-1` ou un `0` dans une colonne numérique. Pandas ne les voit pas, ton œil doit les voir.

### Exercice 4.1 — Traquer les manquants déguisés

1. Vérifie les valeurs manquantes déclarées (`.isna().sum().sum()`) et les doublons.
2. Pour chaque colonne de type texte, compte le nombre et le pourcentage de `"unknown"`.
3. Compte combien de clients ont `pdays == -1`.

Indice pour le point 2 : récupère d'abord les colonnes non numériques avec
`X.select_dtypes(include="number").columns`, puis prends celles qui n'y sont pas.

In [ ]:
# A TOI DE JOUER
# 1. Nombre total de valeurs manquantes et nombre de lignes dupliquees
# 2. Pour chaque colonne texte : nombre et pourcentage de "unknown"
# 3. Nombre et pourcentage de clients avec pdays == -1

**Ce que tu devrais observer** : aucune valeur manquante déclarée, aucun doublon... mais
quatre colonnes contiennent des `unknown` :

| Colonne | `unknown` | Que faire ? |
|---|---|---|
| `job` | 288 (0,6 %) | négligeable, on garde la modalité |
| `education` | 1 857 (4,1 %) | on garde : « niveau d'études non renseigné » est une information en soi |
| `contact` | 13 020 (28,8 %) | on garde : c'est un ancien mode de saisie, et il est peut-être lié au résultat |
| `poutcome` | 36 959 (81,7 %) | on garde : cela signifie « aucune campagne précédente » |

Le réflexe du débutant est de supprimer ces lignes : ici, cela reviendrait à jeter 82 % du
jeu de données. Le bon réflexe est de se demander **pourquoi** la valeur est absente. Pour
`poutcome`, l'absence n'est pas une erreur de saisie : c'est un fait (ce client est un
nouveau prospect). On la traite donc comme une catégorie à part entière.

Et 36 954 clients (81,7 %) ont `pdays = -1`, ce qui est cohérent : ce sont les mêmes clients,
ceux qui n'ont jamais été contactés auparavant.

### Exercice 4.2 — Traiter la valeur sentinelle `-1`

Le `-1` de `pdays` est dangereux : c'est un **code**, pas une durée. Un modèle linéaire, lui,
le lira comme « il y a -1 jour », c'est-à-dire une valeur toute proche de « contacté hier ».
Alors que ces clients sont à l'opposé : ils n'ont **jamais** été contactés.

La solution standard, en deux temps :

1. créer une colonne `deja_contacte` qui vaut 1 si `pdays != -1`, 0 sinon (on garde
   l'information) ;
2. remplacer le `-1` par 0 dans `pdays` (on neutralise le code).

Fais-le sur une copie nommée `X_propre`, puis construis les deux listes de colonnes,
`col_num` et `col_cat` : tu en auras besoin à l'étape 5.

In [ ]:
# A TOI DE JOUER
# 1. X_propre = copie de X
# 2. Ajoute la colonne deja_contacte (1 si pdays != -1, sinon 0)
# 3. Remplace les -1 de pdays par 0
# 4. Construis col_num (colonnes numeriques) et col_cat (les autres), puis affiche-les

**Ce que tu devrais observer** : 7 colonnes numériques (dont la nouvelle `deja_contacte`) et
8 colonnes catégorielles. `pdays` ne descend plus en dessous de 0.

Ce geste porte un nom : c'est du **feature engineering**. On n'a pas ajouté de données, on a
rendu lisible par le modèle une information qui était codée en douce.

---
## Étape 5 — Ton premier modèle

Comme dans le TP de régression, on découpe en trois : **entraînement** (64 %),
**validation** (16 %) et **test** (20 %), ce dernier mis au coffre jusqu'à l'étape 8.

Une précaution supplémentaire ici : avec seulement 11,7 % de « oui », un tirage malchanceux
pourrait déséquilibrer les jeux. On utilise donc `stratify=y`, qui garantit **la même
proportion de souscriptions dans chaque morceau**.

### Exercice 5.1 — Découper en préservant les proportions

Découpe `X_propre` / `y` en test (20 %), puis le reste en apprentissage / validation (20 %),
avec `random_state=42` et `stratify` à chaque fois. Vérifie ensuite le taux de souscription
dans chacun des trois jeux.

Attention : pour le second découpage, `stratify` doit porter sur `y_train`, pas sur `y`.

In [ ]:
# A TOI DE JOUER
# 1. Importe train_test_split
# 2. X_train, X_test, y_train, y_test : 20 % de test, random_state=42, stratify=y
# 3. X_ap, X_val, y_ap, y_val : 20 % de validation, random_state=42, stratify=y_train
# 4. Affiche la taille et le taux de souscription des trois jeux

**Ce que tu devrais observer** : 28 934 clients pour apprendre, 7 234 pour valider, 9 043 au
coffre — et **11,7 % de souscriptions dans chacun des trois**. C'est exactement ce que fait
`stratify` : sans lui, ces taux varieraient d'un jeu à l'autre, et les scores deviendraient
incomparables.

### Exercice 5.2 — Un `Pipeline` : préparer et entraîner en un seul objet

La régression logistique ne sait traiter que des nombres. Il faut donc, avant elle :

- **encoder les catégories** avec un `OneHotEncoder` : la colonne `marital` devient trois
  colonnes 0/1 (`marital_married`, `marital_single`, `marital_divorced`). On passe
  `handle_unknown="ignore"` pour qu'une modalité jamais vue à l'entraînement ne fasse pas
  planter la prédiction ;
- **standardiser les colonnes numériques** avec un `StandardScaler`, pour que `balance` (des
  milliers d'euros) ne domine pas `age` (des dizaines d'années).

Le `ColumnTransformer` applique chaque traitement aux bonnes colonnes, et le `Pipeline`
enchaîne préparation et modèle dans un seul objet.

**Pourquoi c'est important :** un `Pipeline` n'apprend ses transformations (les moyennes du
scaler, les modalités de l'encodeur) que sur les données d'entraînement, à chaque `fit`. En
cross-validation, cela évite qu'une information du pli d'évaluation ne se glisse dans la
préparation. C'est la protection anti-fuite la plus efficace du métier — et, accessoirement,
un seul objet à sauvegarder pour la mise en production.

In [ ]:
# A TOI DE JOUER
# 1. Importe Pipeline, ColumnTransformer, OneHotEncoder, StandardScaler,
#    LogisticRegression et DummyClassifier
# 2. preparation = ColumnTransformer qui applique StandardScaler aux col_num
#    et OneHotEncoder(handle_unknown="ignore") aux col_cat
# 3. modele = Pipeline([("preparation", preparation), ("modele", LogisticRegression(max_iter=1000))])
# 4. Entraine modele et naif (DummyClassifier strategy="most_frequent") sur X_ap, y_ap
# 5. Affiche les probabilites predites pour les 5 premiers clients de validation

**Ce que tu devrais observer** : les 15 colonnes de départ sont devenues **51 colonnes**
après encodage — c'est le `OneHotEncoder` qui a éclaté chaque catégorie.

Surtout, regarde les probabilités : le modèle ne répond pas « oui » ou « non », il donne une
**probabilité de souscription** pour chaque client (par exemple 0,03, 0,12, 0,41...). C'est
`predict` qui tranche ensuite, en appliquant le seuil par défaut de **0,5**. Ce seuil est un
choix, pas une vérité — on le remettra en cause à l'étape 6.

---
## Étape 6 — Évaluer le modèle

**Question de réflexion :** un modèle qui prédirait « personne ne souscrit » pour les
45 000 clients aurait 88,3 % de bonnes réponses. Est-ce un bon modèle ?

### Exercice 6.1 — L'accuracy, et pourquoi elle ment ici

Calcule l'accuracy (la proportion de prédictions correctes) du modèle naïf et de la
régression logistique sur le jeu de validation, puis compte combien de souscripteurs chacun
a identifiés.

Indice : `accuracy_score(y_val, modele.predict(X_val))`, et
`modele.predict(X_val).sum()` donne le nombre de clients que le modèle désigne comme
souscripteurs.

In [ ]:
# A TOI DE JOUER
# 1. Importe accuracy_score depuis sklearn.metrics
# 2. Affiche l'accuracy du modele naif et celle de la regression logistique sur la validation
# 3. Affiche, pour chacun, le nombre de clients qu'il designe comme souscripteurs
#    (a comparer au nombre reel de souscripteurs : y_val.sum())

**Ce que tu devrais observer** :

| Modèle | Accuracy | Clients désignés |
|---|---|---|
| modèle naïf | 0,883 | 0 |
| régression logistique | 0,894 | 222 |

Le modèle naïf obtient **88,3 % de bonnes réponses en ne désignant personne**. Il se contente
de répondre « non » 7 234 fois, et il a raison 88,3 % du temps, puisque 88,3 % des clients ne
souscrivent pas. Un score de 88 % qui ne sert strictement à rien.

Et notre modèle ? 89,4 %, à peine mieux. Pourtant il a identifié 222 clients. **L'accuracy
est incapable de faire la différence entre ces deux modèles** : c'est la métrique la plus
trompeuse dès que les classes sont déséquilibrées. Il faut regarder *quelles* erreurs sont
commises.

### Exercice 6.2 — La matrice de confusion, precision et recall

La matrice de confusion croise la réalité et la prédiction :

|  | prédit : non | prédit : oui |
|---|---|---|
| **réellement non** | vrais négatifs (VN) | **faux positifs (FP)** : on appelle pour rien |
| **réellement oui** | **faux négatifs (FN)** : on rate un client | vrais positifs (VP) |

Deux métriques en découlent, et elles répondent à deux questions métier différentes :

- **Precision** = VP / (VP + FP) — *quand le modèle dit « appelle-le », a-t-il raison ?*
  C'est le rendement des appels passés.
- **Recall** = VP / (VP + FN) — *quelle part des souscripteurs le modèle a-t-il retrouvés ?*
  C'est le chiffre d'affaires capté.
- **F1-score** : la moyenne harmonique des deux, pour résumer d'un chiffre.

Affiche la matrice de confusion de la régression logistique, puis ses precision, recall et
F1.

In [ ]:
# A TOI DE JOUER
# 1. Importe ConfusionMatrixDisplay, precision_score, recall_score, f1_score
# 2. Affiche la matrice de confusion du modele sur la validation
# 3. Affiche precision, recall et F1 du modele

**Ce que tu devrais observer** : la matrice donne 6 317 VN, 71 FP, 695 FN et 151 VP.
Traduction en français :

- **Precision 0,680** : sur les 222 clients que le modèle désigne, 151 souscrivent vraiment.
  Deux appels sur trois aboutissent, contre un sur huit en appelant au hasard. Excellent.
- **Recall 0,178** : mais sur les 846 souscripteurs présents, le modèle n'en retrouve que
  151. **Il en laisse passer 695.**

Le modèle est donc très prudent : il ne se mouille que lorsqu'il est certain. C'est le seuil
de 0,5 qui produit ce comportement, et pour une campagne marketing, c'est probablement trop
timide : un appel coûte quelques euros, une souscription en rapporte beaucoup plus. Il vaut
mieux accepter des appels inutiles que de rater des clients.

Retiens surtout : **precision et recall varient en sens inverse**, et choisir entre les deux
n'est pas une décision technique, c'est une décision métier.

### Exercice 6.3 — Le seuil de décision, l'AUC, et le vrai livrable

Le modèle produit une probabilité ; le seuil transforme cette probabilité en décision. Ici,
le service marketing ne veut pas vraiment une réponse oui/non : il veut une **liste de
clients à appeler, classée par ordre de priorité**. Notre probabilité fait exactement cela.

1. Construis un tableau donnant, pour les seuils 0,5 / 0,3 / 0,2 / 0,1 : le nombre de
   clients appelés, la precision et le recall.
2. Calcule l'AUC (`roc_auc_score`) et trace la courbe ROC.
3. Classe les clients de la validation par probabilité décroissante et regarde ce que donne
   le fait de n'appeler que les **20 % les mieux notés**.

Indices : `probas_val = modele.predict_proba(X_val)[:, 1]`, puis
`(probas_val >= seuil).astype(int)`. Pour le classement : `np.argsort(-probas_val)`.

In [ ]:
# A TOI DE JOUER
# 1. Tableau : pour chaque seuil [0.5, 0.3, 0.2, 0.1], nombre d'appels, precision, recall
# 2. AUC et courbe ROC (RocCurveDisplay.from_predictions)
# 3. Si on n'appelle que les 20 % de clients les mieux notes :
#    combien de souscripteurs capte-t-on, et quel est le taux de reussite des appels ?

**Ce que tu devrais observer** :

| Seuil | Clients appelés | Precision | Recall |
|---|---|---|---|
| 0,5 | 222 | 0,680 | 0,178 |
| 0,3 | 509 | 0,552 | 0,332 |
| 0,2 | 819 | 0,453 | 0,439 |
| 0,1 | 2 782 | 0,218 | 0,717 |

Le seuil est **le levier de pilotage de la campagne**. À 0,5, on passe 222 appels très
rentables mais on capte 18 % du potentiel. À 0,1, on passe douze fois plus d'appels et on
capte 72 % des souscripteurs, avec un rendement divisé par trois. Le bon seuil dépend du coût
d'un appel et de la marge d'une souscription — c'est au marketing de trancher, pas au modèle.

L'**AUC vaut 0,767**. Elle se lit ainsi : si on tire au hasard un souscripteur et un
non-souscripteur, le modèle donne une probabilité plus élevée au bon dans 76,7 % des cas.
Elle a un gros avantage : elle mesure la **qualité du classement**, indépendamment du seuil.
0,5 = hasard, 1 = parfait.

Et la conclusion qui intéresse vraiment le directeur marketing : **en n'appelant que les 20 %
de clients les mieux notés (1 446 appels), on capte 474 souscripteurs sur 846, soit 56 % du
potentiel, avec un appel sur trois qui aboutit** (32,8 %, contre 11,7 % aujourd'hui).

---
## Étape 7 — La cross-validation

**Question de réflexion :** tous ces chiffres viennent d'un seul jeu de validation, tiré au
hasard. Combien vaudraient-ils avec un autre tirage ?

La cross-validation répond à cette question : on découpe le jeu d'entraînement en 5 plis, on
entraîne 5 fois en laissant chaque fois un pli de côté pour l'évaluation, et on obtient
5 scores — donc une moyenne **et** un écart-type.

En classification déséquilibrée, on utilise `StratifiedKFold` plutôt que `KFold` : il
garantit les mêmes 11,7 % de souscriptions dans chaque pli, pour la même raison que
`stratify` à l'étape 5.

### Exercice 7.1 — Cross-validation en 5 plis stratifiés

Lance une cross-validation du `Pipeline` sur `X_train` / `y_train`, avec
`StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` et les métriques
`["accuracy", "precision", "recall", "f1", "roc_auc"]`. Affiche la moyenne et l'écart-type de
chacune.

Attention : passe un `Pipeline` **non entraîné** à `cross_validate`, qui se charge de le
ré-entraîner sur chaque pli. C'est précisément ce qui garantit l'absence de fuite.

In [ ]:
# A TOI DE JOUER
# 1. Importe cross_validate et StratifiedKFold
# 2. Cree un nouveau Pipeline identique a celui de l'etape 5 (non entraine)
# 3. Lance la cross-validation en 5 plis stratifies sur X_train / y_train
# 4. Affiche moyenne et ecart-type de chaque metrique

**Ce que tu devrais observer** :

| Métrique | Moyenne | Écart-type |
|---|---|---|
| accuracy | 0,892 | 0,001 |
| precision | 0,643 | 0,014 |
| recall | 0,175 | 0,011 |
| f1 | 0,275 | 0,013 |
| roc_auc | 0,763 | 0,003 |

Deux lectures :

1. **Les estimations sont stables.** L'AUC bouge de ±0,003 d'un pli à l'autre : on peut
   annoncer 0,76 sans trembler. La precision, elle, varie de ±0,014, car elle se calcule sur
   les quelques centaines de clients désignés — moins on mesure sur de monde, plus le chiffre
   danse.
2. **L'accuracy est d'un ennui total** : 0,892 ± 0,001, toujours collée aux 88,3 % du modèle
   naïf. Elle ne distinguerait pas un bon modèle d'un mauvais. Sur ce projet, la métrique à
   suivre est l'**AUC** (qualité du classement), puis la precision et le recall une fois le
   seuil choisi.

---
## Étape 8 — Verdict final sur le jeu de test

Le jeu de test n'a pas été touché depuis l'étape 5. On ré-entraîne le modèle retenu sur
**tout** le jeu d'entraînement (apprentissage + validation), puis on l'évalue **une seule
fois**.

### Exercice 8.1 — L'épreuve de vérité, et le chiffrage métier

1. Entraîne un `Pipeline` neuf sur `X_train` / `y_train` en entier.
2. Sur le jeu de test : accuracy, precision, recall, F1, AUC, et matrice de confusion.
3. Compare l'AUC obtenue à celle annoncée par la cross-validation.
4. Chiffre le gain : si le marketing n'appelle que les 20 % de clients les mieux notés du
   jeu de test, combien d'appels, combien de souscripteurs, et quel taux de réussite ?

In [ ]:
# A TOI DE JOUER
# 1. modele_final : un Pipeline neuf entraine sur X_train / y_train
# 2. Toutes les metriques sur le jeu de test + matrice de confusion
# 3. Comparaison avec l'annonce de la cross-validation (AUC)
# 4. Scenario : appeler les 20 % de clients les mieux notes du jeu de test

**Ce que tu devrais observer** :

| Métrique | Cross-validation | Jeu de test |
|---|---|---|
| AUC | 0,763 ± 0,003 | **0,772** |
| precision | 0,643 | 0,664 |
| recall | 0,175 | 0,180 |

La cross-validation annonçait 0,763 d'AUC, le test donne 0,772 : la promesse est tenue. Le
modèle se comporte sur des clients inconnus comme il se comportait à l'entraînement.

Et le chiffrage, qui est le vrai livrable :

> Campagne actuelle : **9 043 appels** pour **1 058 souscriptions** (11,7 % de réussite).
> Campagne ciblée sur les 20 % les mieux notés : **1 808 appels** pour **588 souscriptions**
> (32,5 % de réussite).
>
> **56 % des souscripteurs captés avec 80 % d'appels en moins.**

---
## La réponse au directeur marketing

> **Oui, on peut cibler.** Le modèle classe les clients par probabilité de souscription et
> atteint une AUC de 0,77, stable d'un échantillon à l'autre. En concentrant la campagne sur
> les 20 % de clients les mieux notés, on conserve **56 % des souscriptions** en passant
> **cinq fois moins d'appels** : un appel sur trois aboutit, contre un sur huit aujourd'hui.
>
> Le curseur reste entre vos mains : appeler plus de clients augmente le nombre de
> souscriptions et fait baisser le rendement de chaque appel. Donnez-nous le coût d'un appel
> et la marge d'une souscription, et on calcule le point d'équilibre.
>
> **Ce que le modèle a appris**, et qui recoupe votre intuition terrain : les meilleurs
> prospects sont les clients ayant déjà souscrit lors d'une campagne précédente (65 % de
> réussite), les plus de 60 ans, les étudiants, et les clients sans crédit immobilier en
> cours.

## À retenir

1. **L'accuracy ment sur des classes déséquilibrées.** Un modèle qui ne désigne personne
   obtient 88,3 %. Toujours comparer à un `DummyClassifier`, et regarder la matrice de
   confusion.
2. **Chaque métrique répond à une question métier.** Precision = rendement des appels,
   recall = chiffre d'affaires capté, AUC = qualité du classement. On choisit la métrique
   avant de modéliser, en fonction du coût des erreurs.
3. **Le seuil est un curseur métier**, pas une constante. 0,5 n'a rien de sacré.
4. **Méfie-toi des variables trop belles.** `duration` aurait fait monter l'AUC à 0,91 et
   donné un modèle inutilisable : elle n'existe qu'après l'appel.
5. **Les manquants ne sont pas toujours déclarés.** `unknown`, `-1`, `0` : il faut les
   traquer à la main, et comprendre *pourquoi* la valeur est absente avant de décider.
6. **Le `Pipeline` n'est pas une coquetterie** : il garantit que l'encodage et la
   standardisation sont appris sur le seul jeu d'entraînement, à chaque pli.

## Pour aller plus loin

Ces pistes n'ont pas de corrigé : c'est à toi de jouer.

1. **Vérifie le piège** : refais tourner la cross-validation en gardant `duration` dans `X`.
   L'AUC monte à environ 0,91. Explique en trois phrases pourquoi ce modèle est pourtant
   inutilisable.
2. **Rééquilibre les classes** : passe `class_weight="balanced"` à la `LogisticRegression`.
   Que deviennent la precision et le recall ? Est-ce vraiment mieux que de baisser le seuil ?
3. **Change de modèle** : remplace la régression logistique par un
   `RandomForestClassifier(n_estimators=200)`. L'AUC en cross-validation progresse-t-elle de
   plus d'un écart-type ?
4. **Chiffre le point d'équilibre** : en supposant 5 € par appel et 150 € de marge par
   souscription, trace le profit en fonction du seuil et trouve le seuil optimal.